# Diabetes Prediction Using Machine Learning

Compare Decision Tree and KNN using the Pima Indians Diabetes Database.

Dataset: https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv

Educational use only; not for medical diagnosis.

In [ ]:
import os
import sys
import json
import zipfile
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

print("=" * 60)
print("DIABETES PREDICTION USING MACHINE LEARNING")
print("=" * 60)

# 1. DOWNLOAD DATASET DIRECTLY FROM GITHUB
dataset_url = (
    "https://raw.githubusercontent.com/"
    "npradaschnor/Pima-Indians-Diabetes-Dataset/"
    "master/diabetes.csv"
)

df = pd.read_csv(dataset_url)

print("\n1. DATASET LOADED SUCCESSFULLY")
print("Dataset shape:", df.shape)
display(df.head())

# 2. INSPECT DATA
print("\n2. DATA INFORMATION")
print("\nColumn names:", df.columns.tolist())
print("\nMissing values before cleaning:")
print(df.isnull().sum())

print("\nTarget distribution:")
print(df["Outcome"].value_counts())

# 3. PREPARE DATA
X = df.drop("Outcome", axis=1).copy()
y = df["Outcome"].copy()

zero_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

X[zero_columns] = X[zero_columns].replace(0, np.nan)

print("\n3. INVALID ZERO MEASUREMENTS")
print("Missing/invalid measurements after checking zeros:")
print(X.isnull().sum())

# Split before imputing to avoid data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

# 4. BUILD TWO MACHINE LEARNING MODELS
decision_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ))
])

knn = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(n_neighbors=5))
])

# 5. TRAIN MODELS
decision_tree.fit(X_train, y_train)
knn.fit(X_train, y_train)

print("\n4. BOTH MODELS TRAINED SUCCESSFULLY")

# 6. PREDICT TEST RESULTS
y_pred_tree = decision_tree.predict(X_test)
y_pred_knn = knn.predict(X_test)

accuracy_tree = accuracy_score(y_test, y_pred_tree)
accuracy_knn = accuracy_score(y_test, y_pred_knn)

print("\n5. MODEL ACCURACY")
print(f"Decision Tree Accuracy: {accuracy_tree:.2%}")
print(f"KNN Accuracy:           {accuracy_knn:.2%}")

results = pd.DataFrame({
    "Model": ["Decision Tree", "KNN"],
    "Accuracy": [accuracy_tree, accuracy_knn]
})

display(results.style.format({"Accuracy": "{:.2%}"}))

# 7. ACCURACY COMPARISON CHART
plt.figure(figsize=(7, 4))
sns.barplot(
    data=results,
    x="Model",
    y="Accuracy",
    hue="Model",
    legend=False
)
plt.title("Diabetes Prediction: Model Accuracy Comparison")
plt.ylabel("Test Accuracy")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

# 8. CONFUSION MATRICES
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_tree,
    labels=[0, 1],
    display_labels=["No Diabetes", "Diabetes"],
    cmap="Blues",
    ax=axes[0]
)
axes[0].set_title("Decision Tree Confusion Matrix")

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_knn,
    labels=[0, 1],
    display_labels=["No Diabetes", "Diabetes"],
    cmap="Greens",
    ax=axes[1]
)
axes[1].set_title("KNN Confusion Matrix")

plt.tight_layout()
plt.show()

# 9. CLASSIFICATION REPORTS
print("\n6. DECISION TREE CLASSIFICATION REPORT")
print(classification_report(
    y_test,
    y_pred_tree,
    target_names=["No Diabetes", "Diabetes"],
    zero_division=0
))

print("\n7. KNN CLASSIFICATION REPORT")
print(classification_report(
    y_test,
    y_pred_knn,
    target_names=["No Diabetes", "Diabetes"],
    zero_division=0
))

# 10. AUTOMATIC CONCLUSION
if accuracy_tree > accuracy_knn:
    winner = "Decision Tree"
elif accuracy_knn > accuracy_tree:
    winner = "KNN"
else:
    winner = "Both models achieved equal accuracy"

print("\n8. PROJECT CONCLUSION")
print("-" * 45)
print(f"Decision Tree Accuracy: {accuracy_tree:.2%}")
print(f"KNN Accuracy:           {accuracy_knn:.2%}")
print("Better model by test accuracy:", winner)

print(
    "\nThe models were evaluated on the same held-out test set. "
    "Accuracy alone does not establish clinical usefulness; "
    "review the confusion matrices and recall as well."
)

# 11. SAVE RESULTS
results.to_csv("model_accuracy_results.csv", index=False)

with open("project_results.txt", "w") as f:
    f.write("DIABETES PREDICTION PROJECT RESULTS\n")
    f.write(f"Decision Tree Accuracy: {accuracy_tree:.2%}\n")
    f.write(f"KNN Accuracy: {accuracy_knn:.2%}\n")
    f.write(f"Better model by test accuracy: {winner}\n")

# 12. CREATE README FILE
readme = f"""# Diabetes Prediction Using Machine Learning

## Project Overview
This project compares Decision Tree and K-Nearest Neighbors (KNN)
models for diabetes classification using health-related measurements.

## Dataset
Pima Indians Diabetes Database:
{dataset_url}

The notebook downloads the dataset directly from this URL.

## Technologies Used
- Python
- Google Colab / Jupyter Notebook
- Pandas and NumPy
- Matplotlib and Seaborn
- Scikit-learn

## Data Preparation
- Checked missing values.
- Treated zero values in selected medical measurement columns
  as missing values.
- Used median imputation inside training pipelines.
- Split data into 80% training and 20% testing.

## Models
- Decision Tree (maximum depth 5)
- K-Nearest Neighbors (5 neighbors)

## Results
- Decision Tree test accuracy: {accuracy_tree:.2%}
- KNN test accuracy: {accuracy_knn:.2%}
- Better by test accuracy: {winner}

See the notebook for confusion matrices and classification reports.

## How to Run
1. Open the notebook in Google Colab.
2. Run its code cell.
3. The dataset is loaded from the URL above.

## Conclusion
The two classifiers were compared using test accuracy and confusion
matrices. The better model by accuracy was: {winner}.

## Disclaimer
This is an educational project, not a medical diagnostic tool.
The dataset represents a specific population and results may not
generalize to other populations.
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)

# 13. CREATE A DOWNLOADABLE NOTEBOOK
# The notebook contains the complete executable project code.
try:
    from google.colab import files

    notebook_code = r'''# This notebook runs the full Diabetes Prediction project.
# Run this cell to download the dataset, train both models,
# display accuracy and confusion matrices, and create project files.
'''

    # Store the full source of this cell in the notebook.
    # IPython's history contains the code currently being executed.
    try:
        from IPython import get_ipython
        notebook_code = get_ipython().history_manager.input_hist_raw[-1]
        if not notebook_code.strip():
            notebook_code = (
                "# Paste the complete project code from the original "
                "Colab cell into this cell if needed.\n"
            )
    except Exception:
        pass

    notebook = {
        "cells": [
            {
                "cell_type": "markdown",
                "metadata": {},
                "source": [
                    "# Diabetes Prediction Using Machine Learning\n",
                    "\n",
                    "Compare Decision Tree and KNN using the Pima Indians "
                    "Diabetes Database.\n",
                    "\n",
                    f"Dataset: {dataset_url}\n",
                    "\n",
                    "Educational use only; not for medical diagnosis."
                ]
            },
            {
                "cell_type": "code",
                "execution_count": None,
                "metadata": {},
                "outputs": [],
                "source": notebook_code.splitlines(keepends=True)
            }
        ],
        "metadata": {
            "kernelspec": {
                "display_name": "Python 3",
                "language": "python",
                "name": "python3"
            },
            "language_info": {"name": "python"}
        },
        "nbformat": 4,
        "nbformat_minor": 5
    }

    with open("Diabetes_Prediction.ipynb", "w", encoding="utf-8") as f:
        json.dump(notebook, f, indent=2)

    # Bundle deliverables for easy download
    with zipfile.ZipFile(
        "Diabetes_Prediction_Submission.zip", "w",
        zipfile.ZIP_DEFLATED
    ) as z:
        z.write("Diabetes_Prediction.ipynb")
        z.write("README.md")
        z.write("model_accuracy_results.csv")
        z.write("project_results.txt")

    print("\n9. PROJECT FILES CREATED")
    print("README.md")
    print("Diabetes_Prediction.ipynb")
    print("model_accuracy_results.csv")
    print("project_results.txt")
    print("Diabetes_Prediction_Submission.zip")
    print("\nDataset link included in README:", dataset_url)

    print(
        "\nDownloading the ZIP now. Extract it, then upload the "
        "notebook and README to your GitHub repository."
    )
    files.download("Diabetes_Prediction_Submission.zip")

except Exception as e:
    print("\nProject results and README were saved in the Colab files.")
    print("Download them from the Files sidebar if automatic download fails.")
    print("Details:", e)

print("\nPROJECT EXECUTION FINISHED")